# 12. Historical Replay Pipeline

Hand-driven verification of the Algorithm 1a (Historical Replay) pipeline implemented per the deep-interview spec at `.omc/specs/deep-interview-historical-replay.md`.

**Sections (single code cell, divided by `# %% N` markers):**

1. Build a resolved bike-sharing model from the mock loader.
2. Compute the per-commodity conservation baseline from `init_state`.
3. Assemble and run the canonical historical-replay phase pipeline.
4. **Cell A** — inventory and flow parity vs the observed tables.
5. **Cell B** — per-commodity conservation invariant across periods.
6. **Cell C** — overflow safety-net stub (skeleton); the canonical pipeline never triggers it because `observed_flow.target_id` is post-redirect.
7. **Cell D** — τ_k handling: same-period vs cross-period arrival distribution.
8. **Cell E** — end-of-period deficit clip: complementary lost-demand mode that pairs `permissive` physics with a final settlement phase. Compares canonical replay (zero lost demand by construction) against an inflated-demand treatment scenario (lost demand > 0).

Cells end with assignments only — inspect the named variables in Jupyter's variable explorer or by typing the name in a fresh cell.

In [ ]:
# %% 1. Build resolved model from the bike-sharing mock loader
import dataclasses

import numpy as np
import pandas as pd

from gbp.build.pipeline import build_model
from gbp.consumers.simulator.built_in_phases import (
    ArrivalsPhase,
    DeparturePhysicsPhase,
    EndOfPeriodDeficitPhase,
    HistoricalLatentDemandPhase,
    HistoricalODStructurePhase,
    HistoricalTripSamplingPhase,
    InvariantCheckPhase,
    LatentDemandInflatorPhase,
    OverflowRedirectPhase,
)
from gbp.consumers.simulator.config import EnvironmentConfig
from gbp.consumers.simulator.engine import Environment
from gbp.consumers.simulator.state import init_state
from gbp.loaders.contracts import GraphLoaderConfig
from gbp.loaders.dataloader_graph import DataLoaderGraph
from gbp.loaders.dataloader_mock import DataLoaderMock
from gbp.consumers.simulator.dispatch_phase import DispatchPhase
from gbp.consumers.simulator.phases import Schedule
from gbp.consumers.simulator.tasks.rebalancer import RebalancerTask

mock_config = {
    "n_stations": 10,
    "n_depots": 2,
    "n_timestamps": 72,
    "time_freq": "h",
    "start_date": "2025-01-01",
    "ebike_fraction": 0.3,
    "depot_capacity": 200,
    "seed": 42,
}
mock_source = DataLoaderMock(mock_config, n_trucks=3, truck_capacity_bikes=20)
graph_loader = DataLoaderGraph(mock_source, GraphLoaderConfig())
raw_model = graph_loader.load()
resolved_with_supply = build_model(raw_model)
# Drop supply: observed_flow already encodes trip events on its target side.
resolved_model = dataclasses.replace(resolved_with_supply, supply=None)

# %% 2. Compute per-commodity invariant baseline from initial state
state_initial = init_state(resolved_model)
inventory_totals_initial = (
    state_initial.inventory.groupby("commodity_category")["quantity"].sum()
)
in_transit_totals_initial = (
    state_initial.in_transit.groupby("commodity_category")["quantity"].sum()
    if not state_initial.in_transit.empty
    else pd.Series(dtype=float)
)
baseline_per_commodity = (
    inventory_totals_initial.add(in_transit_totals_initial, fill_value=0.0)
    .to_dict()
)

rebalance_every_n_periods = 6
rebalancer_commodity = "electric_bike"
rebalancer_task = RebalancerTask(
    min_threshold=0.3,
    max_threshold=0.7,
    time_limit_seconds=5,
    commodity_category=rebalancer_commodity,
)

# %% 3. Run canonical historical-replay pipeline
# fail_on_violation=False in the verify notebook lets violations land in the
# log table for inspection rather than aborting on the first deviation; the
# canonical-production setting per ADR is True.
phases_canonical = [
    HistoricalLatentDemandPhase(),
    HistoricalODStructurePhase(),
    DeparturePhysicsPhase(mode="permissive"),
    HistoricalTripSamplingPhase(use_durations=True),
    ArrivalsPhase(),
    OverflowRedirectPhase(),
    DispatchPhase(
        rebalancer_task,
        schedule=Schedule.every_n(rebalance_every_n_periods),
    ),
    InvariantCheckPhase(
        baseline=baseline_per_commodity,
        fail_on_violation=False,
    ),
]
env_canonical = Environment(
    resolved_model,
    EnvironmentConfig(
        phases=phases_canonical, seed=42, scenario_id="historical_replay",
    ),
)
env_canonical.run()
log_canonical = env_canonical.log.to_dataframes()

# %% 4. Cell A — Inventory and flow parity vs observed tables
# Restrict the inventory parity check to facility_ids that observed_inventory
# actually covers — telemetry on the bike-share fixture excludes depots, so
# including them would surface spurious deltas of full depot inventory.
inventory_keys = ["period_id", "facility_id", "commodity_category"]
observed_facility_ids = set(resolved_model.observed_inventory["facility_id"].unique())
inventory_observed = (
    resolved_model.observed_inventory
    .groupby(inventory_keys, as_index=False)["quantity"]
    .last()
    .rename(columns={"quantity": "quantity_observed"})
)
inventory_simulated_full = (
    log_canonical["simulation_inventory_log"]
    .groupby(inventory_keys, as_index=False)["quantity"]
    .last()
    .rename(columns={"quantity": "quantity_simulated"})
)
inventory_simulated = inventory_simulated_full[
    inventory_simulated_full["facility_id"].isin(observed_facility_ids)
]
inventory_parity = inventory_observed.merge(
    inventory_simulated, on=inventory_keys, how="outer",
).fillna(0.0)
inventory_parity["delta"] = (
    inventory_parity["quantity_simulated"]
    - inventory_parity["quantity_observed"]
)

flow_keys = ["period_id", "source_id", "target_id", "commodity_category"]
flow_observed = (
    resolved_model.observed_flow
    .groupby(flow_keys, as_index=False)["quantity"]
    .sum()
    .rename(columns={"quantity": "quantity_observed"})
)
flow_simulated = (
    log_canonical["simulation_flow_log"]
    .groupby(flow_keys, as_index=False)["quantity"]
    .sum()
    .rename(columns={"quantity": "quantity_simulated"})
)
flow_parity = flow_observed.merge(
    flow_simulated, on=flow_keys, how="outer",
).fillna(0.0)
flow_parity["delta"] = (
    flow_parity["quantity_simulated"] - flow_parity["quantity_observed"]
)

# %% 5. Cell B — Per-commodity conservation across periods
inventory_log = log_canonical["simulation_inventory_log"]
inventory_per_commodity_per_period = (
    inventory_log.groupby(["period_id", "commodity_category"], as_index=False)["quantity"]
    .sum()
    .pivot_table(
        index="period_id", columns="commodity_category",
        values="quantity", aggfunc="sum",
    )
    .fillna(0.0)
)
invariant_violation_log = log_canonical["simulation_invariant_violation_log"]
invariant_constancy = {
    "per_period_totals": inventory_per_commodity_per_period,
    "baseline_per_commodity": baseline_per_commodity,
    "violations": invariant_violation_log,
}

# %% 6. Cell C — Overflow safety-net status (skeleton)
# OverflowRedirectPhase ships as a skeleton: it reads operation_capacity[storage]
# and detects overflow per (facility, commodity), but the per-iteration
# nearest-with-capacity arg-min is a TODO (see class docstring).  On the
# canonical replay (where target_id is post-redirect) no overflow exists,
# so the redirect log is empty and the skeleton is fully exercised.
redirect_log = log_canonical["simulation_redirected_flow_log"]
overflow_safety_net = {
    "redirect_log_rows": len(redirect_log),
    "redirect_log_columns": list(redirect_log.columns),
    "is_canonical_no_op": redirect_log.empty,
}

# %% 7. Cell D — τ_k arrival distribution (same-period vs cross-period)
# Inspect how observed_flow durations spread arrival_period across periods.
# A trip with duration_hours=0 lands the same period; durations > period_dur_h
# push arrival_period beyond period_index, producing in_transit entries.
observed_flow_durations = resolved_model.observed_flow[
    ["period_id", "source_id", "target_id", "commodity_category",
     "quantity", "duration_hours"]
].copy()
duration_distribution = (
    observed_flow_durations["duration_hours"]
    .describe()
    .to_dict()
)
tau_demonstration = {
    "observed_flow_with_durations": observed_flow_durations,
    "duration_distribution": duration_distribution,
}

# %% 8. Cell E — End-of-period deficit clip (complementary lost-demand mode)
# Pairs DeparturePhysicsPhase(mode="permissive") with EndOfPeriodDeficitPhase.
# Departures are subtracted without start-of-period clipping; arrivals land;
# whatever inventory remains negative at the end of the period is recorded as
# lost_demand and clipped to zero.  Strict mode (start-of-period clip) is the
# upper-bound counterpart of this lower-bound model.
#
# Two pipelines are run on the same fixture:
#   - eop_canonical:   no inflator, multiplier=1.0 implicit. Deficits cannot
#                      arise because end_inventory matches observed_inventory
#                      (non-negative by construction).  Expected: empty
#                      simulation_lost_demand_log.
#   - eop_treatment:   LatentDemandInflatorPhase(multiplier=2.0) inserted
#                      before DeparturePhysicsPhase. The inflator scales the
#                      latent marginals consumed by the physics phase but not
#                      the raw observed_flow quantities used by trip sampling
#                      and arrivals, so the asymmetry surfaces as end-of-
#                      period deficits at net-outflow stations.
phases_eop_canonical = [
    HistoricalLatentDemandPhase(),
    HistoricalODStructurePhase(),
    DeparturePhysicsPhase(mode="permissive"),
    HistoricalTripSamplingPhase(use_durations=True),
    ArrivalsPhase(),
    OverflowRedirectPhase(),
    EndOfPeriodDeficitPhase(),
]
env_eop_canonical = Environment(
    resolved_model,
    EnvironmentConfig(
        phases=phases_eop_canonical, seed=42, scenario_id="eop_canonical",
    ),
)
env_eop_canonical.run()
log_eop_canonical = env_eop_canonical.log.to_dataframes()

phases_eop_treatment = [
    HistoricalLatentDemandPhase(),
    HistoricalODStructurePhase(),
    LatentDemandInflatorPhase(multiplier=2.0),
    DeparturePhysicsPhase(mode="permissive"),
    HistoricalTripSamplingPhase(use_durations=True),
    ArrivalsPhase(),
    OverflowRedirectPhase(),
    EndOfPeriodDeficitPhase(),
]
env_eop_treatment = Environment(
    resolved_model,
    EnvironmentConfig(
        phases=phases_eop_treatment, seed=42, scenario_id="eop_treatment",
    ),
)
env_eop_treatment.run()
log_eop_treatment = env_eop_treatment.log.to_dataframes()

lost_demand_canonical = log_eop_canonical["simulation_lost_demand_log"]
lost_demand_treatment = log_eop_treatment["simulation_lost_demand_log"]
end_of_period_deficit_demo = {
    "canonical_lost_rows": len(lost_demand_canonical),
    "canonical_total_lost": (
        float(lost_demand_canonical["lost"].sum())
        if not lost_demand_canonical.empty
        else 0.0
    ),
    "treatment_lost_rows": len(lost_demand_treatment),
    "treatment_total_lost": (
        float(lost_demand_treatment["lost"].sum())
        if not lost_demand_treatment.empty
        else 0.0
    ),
    "treatment_per_facility": (
        lost_demand_treatment
        .groupby(["facility_id", "commodity_category"], as_index=False)["lost"]
        .sum()
        .sort_values("lost", ascending=False)
        if not lost_demand_treatment.empty
        else lost_demand_treatment
    ),
}

In [4]:
log_canonical.keys()

dict_keys(['simulation_inventory_log', 'simulation_flow_log', 'simulation_resource_log', 'simulation_unmet_demand_log', 'simulation_rejected_dispatches_log', 'simulation_latent_demand_log', 'simulation_lost_demand_log', 'simulation_dock_blocking_log', 'simulation_redirected_flow_log', 'simulation_invariant_violation_log'])